# M11ef: Fee-Aware Ensemble Kelly on QC Cloud

**Model**: Multi-sleeve ensemble (K60 + K30 + VTH) with fee-robust Kelly sizing
**Local verdict**: BEATS K60-vs-BH at p=0.045 @50bps (sign-test, fee sweep survives to ~100bps)
**Goal**: Reproduce M11ef fee-robust ensemble verdict on QC Cloud data.

## Architecture

```
Ensemble = equal-weight of:
  K60: Kelly cap=0.6, threshold-band signal
  K30: Kelly cap=0.3, conservative
  VTH: Vol-targeting sleeve (target vol = rolling 60d realized)

Fee sweep: [0, 5, 10, 25, 50, 100] bps
Rebalance: monthly
Coins: BTC, ETH, SOL, LTC
```

In [ ]:
# Section 1: Setup — QC Cloud QuantBook
from AlgorithmImports import *
from QuantConnect.Research import *
import numpy as np
import pandas as pd
from scipy.stats import binomtest

qb = QuantBook()
print(f"QC Cloud ready — User: {qb.UserId}")

In [ ]:
# Section 2: Load crypto daily data
coins = ["BTCUSD", "ETHUSD", "SOLUSD", "LTCUSD"]
crypto_data = {}

for symbol_str in coins:
    try:
        crypto = qb.AddCrypto(symbol_str, Resolution.Daily, Market.Bitstamp)
        history = qb.History(crypto.Symbol, timedelta(days=3650), Resolution.Daily)
        if len(history) > 0:
            crypto_data[symbol_str] = history
            print(f"{symbol_str}: {len(history)} daily bars loaded")
        else:
            crypto = qb.AddCrypto(symbol_str, Resolution.Daily, Market.Coinbase)
            history = qb.History(crypto.Symbol, timedelta(days=3650), Resolution.Daily)
            crypto_data[symbol_str] = history
            print(f"{symbol_str}: {len(history)} daily bars (Coinbase)")
    except Exception as e:
        print(f"{symbol_str}: unavailable — {e}")

print(f"\nTotal coins loaded: {len(crypto_data)}")

In [ ]:
# Section 3: Kelly threshold-band position sizing
def threshold_band_signal(close_prices, window=60):
    """Generate signal: long if price > rolling mean, short otherwise."""
    prices = pd.Series(close_prices)
    rolling_mean = prices.rolling(window).mean()
    signal = np.where(prices > rolling_mean, 1.0, 0.0)
    return signal

def kelly_sleeve(close_prices, cap=0.6, window=60, fee_bps=50):
    """Single Kelly sleeve with threshold-band signal."""
    prices = pd.Series(close_prices)
    returns = prices.pct_change().dropna().values
    signal = threshold_band_signal(close_prices, window)
    signal = signal[-len(returns):]  # align
    
    # Rolling vol for Kelly sizing
    rolling_vol = pd.Series(returns).rolling(window).std().values * np.sqrt(252)
    rolling_vol = np.where(rolling_vol < 0.01, 0.2, rolling_vol)  # floor
    
    # Kelly weight: cap * (signal / vol)
    kelly_w = np.minimum(cap / rolling_vol, cap) * signal
    
    # Net returns
    weighted_returns = kelly_w * returns
    
    # Fee deduction
    fee = fee_bps / 10000
    trades = np.abs(np.diff(np.concatenate([[0], kelly_w])))
    fee_costs = np.concatenate([[0], trades * fee])[:len(weighted_returns)]
    net_returns = weighted_returns - fee_costs
    
    sharpe = np.mean(net_returns) / (np.std(net_returns) + 1e-10) * np.sqrt(252)
    return sharpe, net_returns

def vol_targeting_sleeve(close_prices, target_vol=0.15, window=60, fee_bps=50):
    """Vol-targeting sleeve: scale position to maintain target volatility."""
    prices = pd.Series(close_prices)
    returns = prices.pct_change().dropna().values
    
    rolling_vol = pd.Series(returns).rolling(window).std().values * np.sqrt(252)
    rolling_vol = np.where(rolling_vol < 0.01, 0.2, rolling_vol)
    
    # Scale to target vol
    scale = target_vol / rolling_vol
    scale = np.minimum(scale, 2.0)  # cap leverage
    
    weighted_returns = scale * returns
    
    fee = fee_bps / 10000
    trades = np.abs(np.diff(np.concatenate([[0], scale])))
    fee_costs = np.concatenate([[0], trades * fee])[:len(weighted_returns)]
    net_returns = weighted_returns - fee_costs
    
    sharpe = np.mean(net_returns) / (np.std(net_returns) + 1e-10) * np.sqrt(252)
    return sharpe, net_returns

print("Sleeve functions defined")

In [ ]:
# Section 4: Individual sleeve backtests
fee_levels = [0, 5, 10, 25, 50, 100]
sleeve_results = []

for symbol, hist in crypto_data.items():
    close = hist["close"].values
    
    for fee in fee_levels:
        # K60
        sharpe_k60, rets_k60 = kelly_sleeve(close, cap=0.6, fee_bps=fee)
        # K30
        sharpe_k30, rets_k30 = kelly_sleeve(close, cap=0.3, fee_bps=fee)
        # VTH
        sharpe_vth, rets_vth = vol_targeting_sleeve(close, target_vol=0.15, fee_bps=fee)
        
        # Buy-and-hold baseline
        bh_returns = np.diff(np.log(close))
        bh_fee = fee / 10000  # one entry fee
        bh_net = bh_returns - bh_fee
        sharpe_bh = np.mean(bh_net) / (np.std(bh_net) + 1e-10) * np.sqrt(252)
        
        sleeve_results.append({
            "coin": symbol,
            "fee_bps": fee,
            "sharpe_k60": sharpe_k60,
            "sharpe_k30": sharpe_k30,
            "sharpe_vth": sharpe_vth,
            "sharpe_bh": sharpe_bh,
        })
        
        print(f"{symbol} @ {fee}bps: K60={sharpe_k60:.3f}, K30={sharpe_k30:.3f}, "
              f"VTH={sharpe_vth:.3f}, BH={sharpe_bh:.3f}")

sleeve_df = pd.DataFrame(sleeve_results)
print(f"\nTotal combos: {len(sleeve_df)}")

In [ ]:
# Section 5: Ensemble (equal-weight) and fee sweep
ensemble_results = []

for symbol, hist in crypto_data.items():
    close = hist["close"].values
    
    for fee in fee_levels:
        _, rets_k60 = kelly_sleeve(close, cap=0.6, fee_bps=fee)
        _, rets_k30 = kelly_sleeve(close, cap=0.3, fee_bps=fee)
        _, rets_vth = vol_targeting_sleeve(close, target_vol=0.15, fee_bps=fee)
        
        min_len = min(len(rets_k60), len(rets_k30), len(rets_vth))
        ensemble_rets = (rets_k60[:min_len] + rets_k30[:min_len] + rets_vth[:min_len]) / 3
        
        sharpe_ensemble = np.mean(ensemble_rets) / (np.std(ensemble_rets) + 1e-10) * np.sqrt(252)
        
        # BH baseline
        bh_returns = np.diff(np.log(close))[:min_len]
        bh_net = bh_returns - fee / 10000
        sharpe_bh = np.mean(bh_net) / (np.std(bh_net) + 1e-10) * np.sqrt(252)
        
        ensemble_results.append({
            "coin": symbol,
            "fee_bps": fee,
            "sharpe_ensemble": sharpe_ensemble,
            "sharpe_bh": sharpe_bh,
            "delta_sharpe": sharpe_ensemble - sharpe_bh,
            "beats_bh": int(sharpe_ensemble > sharpe_bh),
        })
        
        print(f"{symbol} @ {fee}bps: Ensemble={sharpe_ensemble:.3f}, BH={sharpe_bh:.3f}, "
              f"delta={sharpe_ensemble - sharpe_bh:+.3f}")

ensemble_df = pd.DataFrame(ensemble_results)
print(f"\nTotal combos: {len(ensemble_df)}")

In [ ]:
# Section 6: Sign-test verdict across fee levels
# Focus: K60 vs BH @50bps (matching local verdict)
focus_fee = 50
focus = ensemble_df[ensemble_df["fee_bps"] == focus_fee]

if len(focus) > 0:
    beats = focus["beats_bh"].sum()
    total = len(focus)
    win_rate = beats / total
    p_value = binomtest(beats, total, 0.5).pvalue
    median_delta = focus["delta_sharpe"].median()
    
    print("=" * 60)
    print(f"M11ef Ensemble vs BH @ {focus_fee}bps — QC Cloud Verdict")
    print(f"={'=' * 60}")
    print(f"Beats BH: {beats}/{total} ({win_rate:.1%})")
    print(f"Sign-test p-value: {p_value:.4f}")
    print(f"Median delta-Sharpe: {median_delta:+.4f}")
    
    if p_value < 0.05 and win_rate >= 0.60:
        print(f"VERDICT: BEATS @ {focus_fee}bps")
    else:
        print(f"VERDICT: NO BEATS @ {focus_fee}bps")

# Fee robustness: at what fee does ensemble stop beating BH?
print(f"\nFee robustness sweep:")
for fee in fee_levels:
    subset = ensemble_df[ensemble_df["fee_bps"] == fee]
    if len(subset) > 0:
        wins = subset["beats_bh"].sum()
        total = len(subset)
        med_ds = subset["delta_sharpe"].median()
        print(f"  {fee:3d} bps: {wins}/{total} beats, median delta-Sharpe={med_ds:+.4f}")

print(f"\nLocal M11ef result: BEATS K60-vs-BH p=0.045 @50bps")

## Interpretation

This notebook reproduces the M11ef ensemble fee-robust verdict on QC Cloud data.

Key comparison with local Python:
1. **Fee robustness**: Local ensemble survives to ~100bps. QC Cloud should match if data quality is comparable.
2. **K60 vs BH**: Local p=0.045 @50bps. QC Cloud needs p<0.05 for BEATS confirmation.
3. **Multi-sleeve diversification**: Equal-weight ensemble reduces single-sleeve blow-up risk.

If QC Cloud confirms BEATS, proceed to Phase B: deploy as `EnsembleM11ef` QC algorithm.